# 08 — Candidate–Hashtag Bipartite Network

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("08_candidate_hashtag_bipartite_network", total_steps=7, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook builds the candidate–hashtag bipartite network. Candidate nodes connect to hashtag nodes when they appear together in the same tweet. This is the strongest discourse association network in the project.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets = pd.read_csv(PROCESSED / "tweets_with_entities.csv", parse_dates=["createdAt"])
tweets = parse_list_columns(tweets, ["candidates_mentioned", "hashtags"])

edges = make_bipartite_edges(tweets["candidates_mentioned"], tweets["hashtags"], left_name="candidate", right_name="hashtag")
edges.to_csv(PROCESSED / "candidate_hashtag_edges_full.csv", index=False)
edges_filtered = edges[edges["weight"] >= 2].copy()
edges_filtered.to_csv(PROCESSED / "candidate_hashtag_edges_filtered_weight2.csv", index=False)
print("Candidate-hashtag edges:", len(edges), "Filtered:", len(edges_filtered))
edges_filtered.head(25)


In [ ]:
progress.step("Purpose")
# Build bipartite graph with node type annotations
B = nx.Graph()
for _, r in edges_filtered.iterrows():
    c = r["candidate"]
    h = r["hashtag"]
    B.add_node(c, node_type="candidate")
    B.add_node(h, node_type="hashtag")
    B.add_edge(c, h, weight=float(r["weight"]))

summary = network_summary(B, "candidate_hashtag_bipartite")
metrics = compute_network_metrics(B)
metrics["node_type"] = metrics["node"].map(nx.get_node_attributes(B, "node_type"))
metrics["community"] = metrics["node"].map(detect_communities(B))
summary.to_csv(TABLES / "08_candidate_hashtag_bipartite_summary.csv", index=False)
metrics.to_csv(PROCESSED / "candidate_hashtag_bipartite_node_metrics.csv", index=False)
nx.write_gexf(B, NETWORKS / "candidate_hashtag_bipartite_network.gexf")
display(summary)
display(metrics.head(20))


In [ ]:
progress.step("Purpose")
# Candidate hashtag diversity
candidate_diversity = edges.groupby("candidate", as_index=False).agg(
    unique_hashtags=("hashtag", "nunique"),
    total_candidate_hashtag_weight=("weight", "sum"),
    strongest_single_hashtag_weight=("weight", "max")
)
candidate_diversity["hashtag_concentration"] = candidate_diversity["strongest_single_hashtag_weight"] / candidate_diversity["total_candidate_hashtag_weight"]
candidate_diversity = candidate_diversity.sort_values("unique_hashtags", ascending=False)
candidate_diversity.to_csv(PROCESSED / "candidate_hashtag_diversity.csv", index=False)

fig = px.bar(candidate_diversity.sort_values("unique_hashtags", ascending=True).tail(30), x="unique_hashtags", y="candidate", orientation="h", title="Candidate hashtag diversity")
fig.update_layout(height=820, yaxis_title="Candidate", xaxis_title="Unique associated hashtags")
save_plotly(fig, INTERACTIVE / "08_candidate_hashtag_diversity.html", FIGURES / "08_candidate_hashtag_diversity.png")
fig.show()


In [ ]:
progress.step("Purpose")
# Candidate-hashtag heatmap for top candidates and hashtags
if not edges.empty:
    top_candidates = candidate_diversity.head(20)["candidate"].tolist()
    top_hashtags = edges.groupby("hashtag")["weight"].sum().sort_values(ascending=False).head(30).index.tolist()
    pivot = edges[edges["candidate"].isin(top_candidates) & edges["hashtag"].isin(top_hashtags)].pivot_table(index="candidate", columns="hashtag", values="weight", fill_value=0)
    plt.figure(figsize=(20, 11))
    sns.heatmap(pivot, cmap="mako", linewidths=.5)
    plt.title("Candidate-hashtag association heatmap")
    plt.xlabel("Hashtag")
    plt.ylabel("Candidate")
    plt.tight_layout()
    plt.savefig(FIGURES / "08_candidate_hashtag_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show()


In [ ]:
progress.step("candidate_nodes = [n for n, d in B.nodes(data=True) if d.get('node_type') == 'candidate']")
# Candidate projection: candidates connected when they share hashtags
candidate_nodes = [n for n, d in B.nodes(data=True) if d.get("node_type") == "candidate"]
C = nx.bipartite.weighted_projected_graph(B, candidate_nodes)
c_metrics = compute_network_metrics(C)
c_metrics.to_csv(PROCESSED / "candidate_projection_shared_hashtags_metrics.csv", index=False)
nx.write_gexf(C, NETWORKS / "candidate_projection_shared_hashtags.gexf")

display(network_summary(C, "candidate_projection_shared_hashtags"))
display(c_metrics.head(20))


In [ ]:
progress.step("save_pyvis_network(B, INTERACTIVE / '08_candidate_hashtag_bipartite_interactive.html', title='Candidate-hashtag bipartit")
save_pyvis_network(B, INTERACTIVE / "08_candidate_hashtag_bipartite_interactive.html", title="Candidate-hashtag bipartite network", node_metrics=metrics, max_nodes=700)
save_pyvis_network(C, INTERACTIVE / "08_candidate_projection_shared_hashtags_interactive.html", title="Candidate projection network based on shared hashtags", node_metrics=c_metrics, max_nodes=100)
print("Saved bipartite and candidate projection interactive maps.")


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
